# Exploitation de données électorales avec Python
## Evaluation intermédiaire Python pour la data science (mi-semestre 2026)

**Données** : Résultats du 1er tour de l'élection présidentielle française 2022  
**Source** : [data.gouv.fr](https://www.data.gouv.fr/fr/datasets/r/182268fc-2103-4bcb-a850-6cf90b02a9eb)

In [15]:
pip install pandas numpy matplotlib geopandas great_tables cartiflette

Note: you may need to restart the kernel to use updated packages.


## Import des librairies

In [16]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
import geopandas as gpd
from great_tables import GT, style, loc, vals
from great_tables.data import islands

# Affichage complet des colonnes
pd.set_option('display.max_columns', None)
pd.set_option('display.float_format', '{:.2f}'.format)

## Chargement des données

In [17]:
df = pd.read_csv(
    'https://www.data.gouv.fr/fr/datasets/r/182268fc-2103-4bcb-a850-6cf90b02a9eb'
)

print(f"Dimensions du dataframe : {df.shape}")
df.head()

Dimensions du dataframe : (528675, 7)


/tmp/ipykernel_17326/2739486222.py:1: DtypeWarning: Columns (4) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv(


,code_departement,libelle_departement,code_commune,libelle_commune,prenom,nom,voix
0,01,Ain,1,L'Abergement-Clémenciat,Nathalie,ARTHAUD,3
1,01,Ain,2,L'Abergement-de-Varey,Nathalie,ARTHAUD,2
2,01,Ain,4,Ambérieu-en-Bugey,Nathalie,ARTHAUD,38
3,01,Ain,5,Ambérieux-en-Dombes,Nathalie,ARTHAUD,8
4,01,Ain,6,Ambléon,Nathalie,ARTHAUD,0


# exploration générale:
 

## Question 1 — Création/mise à jour des variables `code_commune` et `candidat`

In [18]:
df['code_departement'].unique()

array(['01', '02', '03', '04', '05', '06', '07', '08', '09', '10', '11',
       '12', '13', '14', '15', '16', '17', '18', '19', '2A', '2B', '21',
       '22', '23', '24', '25', '26', '27', '28', '29', '30', '31', '32',
       '33', '34', '35', '36', '37', '38', '39', '40', '41', '42', '43',
       '44', '45', '46', '47', '48', '49', '50', '51', '52', '53', '54',
       '55', '56', '57', '58', '59', '60', '61', '62', '63', '64', '65',
       '66', '67', '68', '69', '70', '71', '72', '73', '74', '75', '76',
       '77', '78', '79', '80', '81', '82', '83', '84', '85', '86', '87',
       '88', '89', '90', '91', '92', '93', '94', '95', '971', '972',
       '973', '974', '976', '988', '987', '975', '986', '977', '978',
       'fr_etranger'], dtype=object)

Il y a un code de departement nommé fr_etranger. Pour donner un sens la variable code commune que nous allons créer on met va la recoder en 99

In [19]:
df["code_departement"] = df["code_departement"].replace({
    "fr_etranger": "99"
})


In [20]:
# Construction du vrai code commune (département + commune sur 3 chiffres)
df['code_commune'] = (
    df['code_departement'].astype(str).str.zfill(2) +
    df['code_commune'].astype(str).str[-3:].str.zfill(3)
)

# Vérification
montrouge = df[df['libelle_commune'].str.contains('Montrouge', case=False, na=False)]
print("Code commune de Montrouge :", montrouge['code_commune'].unique())

rambouillet = df[df['libelle_commune'].str.contains('Rambouillet', case=False, na=False)]
print("Code commune de Rambouillet :", rambouillet['code_commune'].unique())

Code commune de Montrouge : ['92049']
Code commune de Rambouillet : ['78517']


Création de la variable candidat avec nom prenom ensemble avec un espace

In [21]:
# Création de la colonne candidat 
df['candidat'] = df['prenom'] + ' ' + df['nom']

# vérification 
df.sample(5)


,code_departement,libelle_departement,code_commune,libelle_commune,prenom,nom,voix,candidat
302622,56,Morbihan,56166,Plouay,Yannick,JADOT,154,Yannick JADOT
78178,22,Côtes-d'Armor,22060,Gausson,Emmanuel,MACRON,107,Emmanuel MACRON
353575,02,Aisne,02765,Vaucelles-et-Beffecourt,Philippe,POUTOU,1,Philippe POUTOU
301080,52,Haute-Marne,52472,Sexfontaines,Yannick,JADOT,1,Yannick JADOT
368498,43,Haute-Loire,43102,Grazac,Philippe,POUTOU,4,Philippe POUTOU


## Question 2 — Nombre de candidats à l'élection présidentielle 2022
On vérifie comment sont notés les votes blancs , nuls etc... puis on compte les candidats en éliminant ces votes

In [22]:
# recherche de l'écriture des votes ne correspondant pas aux candidats
df['nom'].unique()

array(['ARTHAUD', 'ROUSSEL', 'MACRON', 'LASSALLE', 'LE PEN', 'ZEMMOUR',
       'MÉLENCHON', 'HIDALGO', 'JADOT', 'PÉCRESSE', 'POUTOU',
       'DUPONT-AIGNAN', 'abstentions', 'blancs', 'nuls'], dtype=object)

In [23]:
# comptage des candidates en éliminant les abstentions etc...
candidats = df[~df['nom'].isin(['abstentions', 'blancs', 'nuls'])]['candidat'].nunique()

print(f"En 2022, il y avait {candidats} candidats à l'élection présidentielle.")

En 2022, il y avait 12 candidats à l'élection présidentielle.


## Question 3 — Scores nationaux de chaque candidat

In [24]:
# On crée une table avec uniquement les vrais candidats 
df_candidats = df[~df['nom'].isin(['abstentions', 'blancs', 'nuls'])].copy()
# On groupe les communes  par candidat et on somme sur le critère candidat 
votes= df_candidats.groupby('candidat', as_index=False)['voix'].sum()
#votes

# On somme sur toutes les voix
total_exprimes = df_candidats['voix'].sum()
#total_exprimes
#On calcule le score et on affiche l'ensemble
votes['score_national'] = votes['voix'] / total_exprimes
#votes

Table 1. Réponse à la question 3

In [30]:
from great_tables import GT, style, loc
# Copie pour l'affichage
table_votes = votes.sort_values('voix', ascending=False).copy()
table_votes['voix'] = table_votes['voix'].apply(lambda x: f"{x:,}".replace(',', ' '))
table_votes['score_national'] = table_votes['score_national'].apply(lambda x: f"{x:.2%}")


display(
    GT(table_votes)
    .tab_header(
        title="Elections ",
        subtitle="Résultats du premier tour du 📅 10 avril 2022"
    )
    .cols_label(
        candidat="Candidat",
        voix="Nombre votes (total)",
        score_national="Score (% votes exprimés)"
    )
)

GT(_tbl_data=                 candidat       voix score_national
1         Emmanuel MACRON  9 783 058         27.85%
5           Marine LE PEN  8 133 828         23.15%
4      Jean-Luc MÉLENCHON  7 712 520         21.95%
11           Éric ZEMMOUR  2 485 226          7.07%
9        Valérie PÉCRESSE  1 679 001          4.78%
10          Yannick JADOT  1 627 853          4.63%
3           Jean LASSALLE  1 101 387          3.13%
2          Fabien ROUSSEL    802 422          2.28%
7   Nicolas DUPONT-AIGNAN    725 176          2.06%
0            Anne HIDALGO    616 478          1.75%
8         Philippe POUTOU    268 904          0.77%
6        Nathalie ARTHAUD    197 094          0.56%, _body=<great_tables._gt_data.Body object at 0x7f0ae11a1130>, _boxhead=Boxhead([ColInfo(var='candidat', type=<ColInfoTypeEnum.default: 1>, column_label='Candidat', column_align='left', column_width=None), ColInfo(var='voix', type=<ColInfoTypeEnum.default: 1>, column_label='Nombre votes (total)', column_align='right', column_width=None), ColInfo(var='score_national', type=<ColInfoTypeEnum.default: 1>, column_label='Score (% votes exprimés)', column_align='right', column_width=None)]), _stub=<great_tables._gt_data.Stub object at 0x7f0b0fd3c3e0>, _spanners=Spanners([]), _heading=Heading(title='Elections ', subtitle='Résultats du premier tour du 📅 10 avril 2022', preheader=None), _stubhead=None, _summary_rows=<great_tables._gt_data.SummaryRows object at 0x7f0ae02cc9e0>, _summary_rows_grand=<great_tables._gt_data.SummaryRows object at 0x7f0ae02cc6b0>, _source_notes=[], _footnotes=[], _styles=[], _locale=<great_tables._gt_data.Locale object at 0x7f0b0fd3c640>, _formats=[], _substitutions=[], _col_merge=[], _options=Options(table_id=OptionsInfo(scss=False, category='table', type='value', value=None), table_caption=OptionsInfo(scss=False, category='table', type='value', value=None), table_width=OptionsInfo(scss=True, category='table', type='px', value='auto'), table_layout=OptionsInfo(scss=True, category='table', type='value', value='fixed'), table_margin_left=OptionsInfo(scss=True, category='table', type='px', value='auto'), table_margin_right=OptionsInfo(scss=True, category='table', type='px', value='auto'), table_background_color=OptionsInfo(scss=True, category='table', type='value', value='#FFFFFF'), table_additional_css=OptionsInfo(scss=False, category='table', type='values', value=[]), table_font_names=OptionsInfo(scss=False, category='table', type='values', value=['-apple-system', 'BlinkMacSystemFont', 'Segoe UI', 'Roboto', 'Oxygen', 'Ubuntu', 'Cantarell', 'Helvetica Neue', 'Fira Sans', 'Droid Sans', 'Arial', 'sans-serif']), table_font_size=OptionsInfo(scss=True, category='table', type='px', value='16px'), table_font_weight=OptionsInfo(scss=True, category='table', type='value', value='normal'), table_font_style=OptionsInfo(scss=True, category='table', type='value', value='normal'), table_font_color=OptionsInfo(scss=True, category='table', type='value', value='#333333'), table_font_color_light=OptionsInfo(scss=True, category='table', type='value', value='#FFFFFF'), table_border_top_include=OptionsInfo(scss=False, category='table', type='boolean', value=True), table_border_top_style=OptionsInfo(scss=True, category='table', type='value', value='solid'), table_border_top_width=OptionsInfo(scss=True, category='table', type='px', value='2px'), table_border_top_color=OptionsInfo(scss=True, category='table', type='value', value='#A8A8A8'), table_border_right_style=OptionsInfo(scss=True, category='table', type='value', value='none'), table_border_right_width=OptionsInfo(scss=True, category='table', type='px', value='2px'), table_border_right_color=OptionsInfo(scss=True, category='table', type='value', value='#D3D3D3'), table_border_bottom_include=OptionsInfo(scss=False, category='table', type='boolean', value=True), table_border_bottom_style=OptionsInfo(scss=True, category='table', type='value', value='solid'), table_border_bottom_width=OptionsInfo(scss=True, c

# Comparaison des scores départements aux moyennes nationales:


## Question 4 — Scores par département

In [32]:
# Agrégation par département et candidat
score_departements = (
    df_candidats
    .groupby(['code_departement', 'candidat'], as_index=False)['voix']
    .sum()
    .rename(columns={'voix': 'votes_departement'})
)

# Calcul du score en % des votes exprimés par département
total_dep = (
    score_departements
    .groupby('code_departement')['votes_departement']
    .sum()
    .rename('total_dep')
)
score_departements = score_departements.merge(total_dep, on='code_departement')
score_departements['score_departement'] = (
    score_departements['votes_departement'] / score_departements['total_dep'] * 100
)
score_departements = score_departements.drop(columns='total_dep')

# Vérification pour l'Aude (département 11)
print("Résultats pour l'Aude (département 11) :")
(
    score_departements[score_departements['code_departement'] == '11']
    .sort_values('votes_departement', ascending=False)
    .assign(score_departement=lambda x: x['score_departement'].map('{:.2f}%'.format))
    .head(12)
)

Résultats pour l'Aude (département 11) :


,code_departement,candidat,votes_departement,score_departement
125,11,Marine LE PEN,64027,30.14%
121,11,Emmanuel MACRON,43104,20.29%
124,11,Jean-Luc MÉLENCHON,42039,19.79%
131,11,Éric ZEMMOUR,18434,8.68%
123,11,Jean LASSALLE,12382,5.83%
129,11,Valérie PÉCRESSE,7350,3.46%
130,11,Yannick JADOT,6322,2.98%
120,11,Anne HIDALGO,6166,2.90%
122,11,Fabien ROUSSEL,5622,2.65%
127,11,Nicolas DUPONT-AIGNAN,4206,1.98%


## Question 5 — Fusion avec les scores nationaux

In [ ]:
# Jointure avec les scores nationaux
score_departements = score_departements.merge(
    table_votes[['candidat', 'voix', 'score']],
    on='candidat',
    how='left'
)

# Vérification pour l'Aude (3 premiers candidats)
print("Vérification pour l'Aude (département 11) :")
(
    score_departements[score_departements['code_departement'] == '11']
    .sort_values('voix', ascending=False)
    .assign(score_departement=lambda x: x['score_departement'].map('{:.2f}%'.format))
    .assign(score_national=lambda x: x['score_national'].map('{:.2f}%'.format))
    .head(3)
    [['code_departement', 'candidat', 'votes_departement', 'score_departement',
      'votes_national', 'score_national']]
)

KeyError: "['votes_national', 'score_national'] not in index"

In [14]:
table_votes

,candidat,voix,score
1,Emmanuel MACRON,9 783 058,27.85%
5,Marine LE PEN,8 133 828,23.15%
4,Jean-Luc MÉLENCHON,7 712 520,21.95%
11,Éric ZEMMOUR,2 485 226,7.07%
9,Valérie PÉCRESSE,1 679 001,4.78%
10,Yannick JADOT,1 627 853,4.63%
3,Jean LASSALLE,1 101 387,3.13%
2,Fabien ROUSSEL,802 422,2.28%
7,Nicolas DUPONT-AIGNAN,725 176,2.06%
0,Anne HIDALGO,616 478,1.75%
